# Plateau’s Problem for Catenoid

This notebook provides an implementation of the Plateau's problem, which finds a minimal surface shape that connects a set of interfaces.
<!-- More details on this example, can be found in [our paper](https://arxiv.org/abs/2402.14009), Sections 4.1 and A.2. -->

### Imports and setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
from tqdm.notebook import trange
import k3d
import sys
import os
import time
import copy
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.model_architecture import GeneralNet
from util.surface_sampling import sample_model_surface_binsearch, sample_model_surface_newton 
from util.error_metrics import chamfer_div, compute_distance 
from util.visualization.utils_mesh import get_mesh
from training.residuals import bind_model, r_data, r_eikonal, r_mean_curvature
from typing import Callable, Union, Optional

torch.manual_seed(0)


device = 'cuda' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

### True Surface

In [ ]:
# Parametric equations for the catenoid in polar coordinates
def catenoid_surface_polar(z, phi, c=1.0):
    x = c * torch.cosh(z / c) * torch.cos(phi)
    y = c * torch.cosh(z / c) * torch.sin(phi)
    return x, y, z

# Level set function for the catenoid
def catenoid_level_set(p, c=1.0):
    x = p[:, 0]
    y = p[:, 1]
    z = p[:, 2]
    return x**2 + y**2 - (c**2 * torch.cosh(z / c)**2)

def sample_true_surface(n_samples, c=1.0):
    z = torch.linspace(-z_max, z_max, n_samples, dtype=torch.float64)
    phi = torch.linspace(-torch.pi, torch.pi, 2*n_samples, dtype=torch.float64)    
    z, phi = torch.meshgrid(z, phi, indexing='ij')
    x, y, z = catenoid_surface_polar(z.flatten(), phi.flatten(), c)
    points_on_surface = torch.vstack([x, y, z]).T
    return points_on_surface

# Bounds and number of samples
n = 1000
phi = torch.linspace(-torch.pi, torch.pi, n, dtype=torch.float64)
z_max = 1.0
c = 1.0

# Generate boundary points for the upper and lower circles
z_constant_upper = torch.full_like(phi, z_max, dtype=torch.float64)
z_constant_lower = torch.full_like(phi, -z_max, dtype=torch.float64)
x_upper, y_upper, z_upper = catenoid_surface_polar(z_constant_upper, phi, c)
x_lower, y_lower, z_lower = catenoid_surface_polar(z_constant_lower, phi, c)
pts_upper = torch.vstack([x_upper, y_upper, z_upper]).T
pts_lower = torch.vstack([x_lower, y_lower, z_lower]).T
pts_boundary = torch.cat([pts_upper, pts_lower], dim=0)
pts_surface_true = sample_true_surface(64)


# Generate the mesh using the level set function
verts, faces = get_mesh(
    lambda x: catenoid_level_set(x, c),
    N=128, 
    device=device,
    bbox_min=torch.tensor([-2, -2, -1.25], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 1.25], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

print("True surface:")
fig.display()

### Pretraining

In [ ]:
model = GeneralNet(ks=[3, 32, 32, 1])
model = model.double()
bind_model(model)
# Generate random training points
num_pretrain_samples = 10000
bounds = torch.tensor([[-1,1], [-1,1], [-1,1]], dtype=torch.float64)
pts_pretrain = torch.rand(num_pretrain_samples, 3, dtype=torch.float64) * (bounds[:,1] - bounds[:,0]) + bounds[:,0]


def pretrain_loss(model, params, pts):
    inputs = pts.to(dtype=torch.float64)
    x, y = pts[:, 0], pts[:, 1]
    targets = x**2 + y**2 - 1
    preds = model(inputs).squeeze(1)
    return 0.5 * (preds - targets).square().mean()

# Pretraining loop
pretrain_optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_pretrain_iters = 1000

for i in range(num_pretrain_iters):
    pretrain_optimizer.zero_grad()
    loss = pretrain_loss(model, model.params, pts_pretrain)
    loss.backward()
    pretrain_optimizer.step()
    
    if i % 100 == 0:
        print(f"Pretrain Iter {i}: Loss= {loss.item():.6f}")

print("Pretraining completed!")

verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-3, -3, -1.5], dtype=torch.float64),
    bbox_max=torch.tensor([3, 3, 1.5], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(pts_surface_true.cpu().detach(), point_size=0.05)
fig.display()


### Main training loop

In [ ]:
from torch.func import vmap, jacrev, jacfwd, functional_call
from training.residuals import adjugate_3x3


def _mean_curv(g, h):
    ## Compute the mean curvature from the Jacobian g and the Hessian h
    ## Static, allows to reuse g, h in the same parent function saving compute
    gHg = torch.einsum('bi,bij,bj->b', g, h, g)
    tr_h = torch.einsum('bii->b', h)
    norm_g = g.square().sum(1).sqrt()
    return -(gHg - norm_g**2 * tr_h) / (2 * norm_g**3)

def _gauss_curv(g, h):
    ## Compute the Gauss curvature from the Jacobian g and the Hessian h
    ## Static, allows to reuse g, h in the same parent function saving compute
    adj = adjugate_3x3(h)
    gAdjg = torch.einsum('bi,bij,bj->b', g, adj, g)
    return gAdjg / g.square().sum(1).pow(2)


class ResidualLibrary:
    r"""
    Implements different residuals of interest amenable to Gauss-Newton: 0.5*\int r(x)^2 dx.
    All residuals share the signature: self, params, point, val -> scalar tensor.
    where val may or may not be used internally as an optional target.
    """
    def __init__(self, model):
        self.model = model

        ## all of the below have the signature 
        ## self.*(params, x) with x being [1, D] for self._func and [B, D] for self.func
        
        ## Single
        self._f        = lambda params, x: functional_call(self.model, params, x.unsqueeze(0)).squeeze(0)
        self._grad_x_f = jacrev(self._f, argnums=1)
        self._hess_x_f = jacfwd(self._grad_x_f, argnums=1)

    def _data(self, params, x, val):
        return (self._f(params, x).squeeze() - val).unsqueeze(-1) ## TODO: do we need the (un)squeezing?

    def _eikonal(self, params, x, val=None):
        return self._grad_x_f(params, x).squeeze(1).square().sum(1).sqrt() - 1 ## we could treat val=1

    def _normal(self, params, x, true_normal):
        g = self._grad_x_f(params, x).squeeze(1)
        pred_normal = g / g.norm(p=2)
        return (torch.dot(pred_normal.squeeze(0), true_normal) - 1).unsqueeze(-1)

    def _laplacian(self, params, x, val=None):
        h = self._hess_x_f(params, x).squeeze(1)
        return torch.einsum('bii->b', h)

    def _mean_curvature(self, params, x, val=None):
        g = self._grad_x_f(params, x).squeeze(1)
        h = self._hess_x_f(params, x).squeeze(1)
        return _mean_curv(g, h)
    
    def _gauss_curvature(self, params, x, val=None):
        g = self._grad_x_f(params, x).squeeze(1)
        h = self._hess_x_f(params, x).squeeze(1)
        return _gauss_curv(g, h)

    def _principal_curvature_1(self, params, x, val=None):
        g = self._grad_x_f(params, x).squeeze(1)
        h = self._hess_x_f(params, x).squeeze(1)
        k_m = _mean_curv(g, h)
        k_g = _gauss_curv(g, h)
        return k_m + torch.sqrt(k_m**2 - k_g)

    def _principal_curvature_2(self, params, x, val=None):
        g = self._grad_x_f(params, x).squeeze(1)
        h = self._hess_x_f(params, x).squeeze(1)
        k_m = _mean_curv(g, h)
        k_g = _gauss_curv(g, h)
        return k_m - torch.sqrt(k_m**2 - k_g)
    
    def _strain(self, params, x, val=None):
        r"""
        Strain is 
        \int k1^2 + k2^2 dx = \int \sqrt{k1^2 + k2^2}^2 dx
        so the residual can be written as r(x) = \sqrt{k1^2 + k2^2}.
        This can be simplified in terms of the mean and Gaussian curvatures:
        k1^2 + k2^2 = 4*k_m^2 - 2*k_g
        """
        g = self._grad_x_f(params, x).squeeze(1)
        h = self._hess_x_f(params, x).squeeze(1)
        k_m = _mean_curv(g, h)
        k_g = _gauss_curv(g, h)
        return torch.sqrt(4*k_m**2 - 2*k_g)

residual_library = ResidualLibrary(model)

## Check usage
# x = pts_boundary[0].float()
# residual_library._data(model.params, x, 0)
# residual_library._eikonal(model.params, x)
# residual_library._normal(model.params, x, torch.tensor([.0, .0, .0]))
# residual_library._laplacian(model.params, x)
# residual_library._strain(model.params, x)

In [ ]:
class ResidualTerm:
    def __init__(
        self, 
        func: Callable, 
        weight: float, 
        points: torch.Tensor, 
        vals: Optional[Union[float, int, torch.Tensor, None]] = None
    ):
        self.func = func
        self.weight = weight
        self.points = points
        self.vals = vals
        
    def vmap(self, func: Callable) -> Callable:
        """
        Vectorizes the callable func(params, points, vals) based on the format of self.vals.
        This is used for both the residual itself and its parameter Jacobian.
        """
        ## Same value for all points (including None).
        ## More scalar cases exist, eg 0-dim torch.Tensor, numpy.ndarray but hard to check for a general scalar-like
        if self.vals is None or isinstance(self.vals, (float, int)):
            return vmap(func, in_dims=(None, 0, None))
        ### Unique value for each point 
        elif isinstance(self.vals, torch.Tensor) and len(self.vals)==len(self.points):
            return vmap(func, in_dims=(None, 0, 0))
        else:
            raise ValueError(f"Invalid format for vals: {type(self.vals)}, shape: {getattr(self.vals, 'shape', None)}")
    
    def eval(self, params) -> torch.Tensor:
        """
        Evaluate the model on the batch of self.points using vmap.
        Returns:
            Tensor of shape [B, 1]: one residual value per point
        """
        return self.vmap(self.func)(params, self.points, self.vals)

    def unweighted_loss(self, params) -> torch.Tensor:
        r"""
        Evaluate the residual 0.5 * \int r(x)^2 dx
        Returns:
            Scalar tensor of shape []
        """
        return 0.5*self.eval(params).square().mean()
    
    def weighted_loss(self, params) -> torch.Tensor:
        """
        Returns:
            Scalar tensor of shape []
        """
        return self.weight * self.unweighted_loss(params)
    
    def grad_theta_r(self, params) -> torch.Tensor:
        """
        Evaluate the Jacobian of the residual on the batch of self.points using vmap.
        Returns:
            dict of Jacobian tensors, for each param tensor the shape is: [B, 1, *param.shape]
        """
        _grad_theta_r = jacrev(self.func, argnums=0)
        return self.vmap(_grad_theta_r)(params, self.points, self.vals) 
    
def compute_loss(params, res_terms, return_unweighted_losses=False):
    unweighted_losses = {key: res_term.unweighted_loss(params) for key, res_term in res_terms.items()} ## might be used for logging
    loss = sum(res_terms[key].weight*unweighted_losses[key] for key in res_terms)
    if return_unweighted_losses:
        return loss, unweighted_losses
    return loss

        
## Example
pts_eikonal = 2 * torch.rand([1000, 3], dtype=torch.float64) - 1
pts_eikonal = torch.cat((pts_eikonal, pts_boundary))

## NOTE: could also subclass dict with compute_loss method
res_terms = {
    "data": ResidualTerm(residual_library._data, 1.0, pts_boundary, vals=0),
    "eikonal": ResidualTerm(residual_library._eikonal, 0.001, pts_eikonal),
    "mean_curvature": ResidualTerm(residual_library._mean_curvature, 1.0, pts_boundary),
}
# class ResidualTermCollection(dict[str, ResidualTerm]):
#     def compute_loss(self, params: torch.Tensor):
#         return sum(rt.weighted_loss(params) for rt in self.values())

#     def compute_loss_verbose(self, params: torch.Tensor):
#         unweighted_losses = {key: rt.unweighted_loss(params) for key, rt in self.items()}
#         loss = sum(rt.weight * unweighted_losses[key] for key, rt in self.items())
#         return loss, unweighted_losses

## Example
# model = model.double()
# params = model.params

# loss, unweighted_losses = compute_loss(params, res_terms, return_unweighted_losses=True)
# print(loss.item())
# for key, l in unweighted_losses.items():
#     print(key, l.item())
    
##
# res_terms["data"].grad_theta_r(params)["fcs.0.weight"].shape
# res_terms["data"].weighted_loss(params)

In [ ]:
def compute_JTJ_per_residual(params, res_term) -> torch.Tensor:
    """Returns a tensor of shape [P,P] where P is the number of parameters"""
    J_dict = res_term.grad_theta_r(params)
    J = torch.cat([p.flatten(start_dim=1) for p in J_dict.values()], dim=1)
    JTJ = torch.einsum('bi,bj->ij', J, J) / len(res_term.points)
    return JTJ.detach()

def compute_JTJ(params, res_terms) -> torch.Tensor:
    """Returns a tensor of shape [P,P] where P is the number of parameters"""
    JTJ = sum(
        res_term.weight * compute_JTJ_per_residual(params, res_term)
        for res_term in res_terms.values()
    )
    return JTJ

## Example
# for key in res_terms:
#     print(compute_JTJ_per_residual(params, res_terms[key]).shape)
# compute_JTJ(params, res_terms)

In [ ]:
from torch.nn.utils import parameters_to_vector, vector_to_parameters

class GaussNewton:
    def __init__(self, model, res_terms, lr=0.1, regularization=1e-6, do_line_search=False, line_search_steps=15):
        self.params_dict = dict(model.named_parameters())
        self.params_list = list(model.parameters())
        self.res_terms = res_terms
        self.lr = lr
        self.regularization = regularization
        self.do_line_search = do_line_search
        self.line_search_steps = line_search_steps
        self.t = 0
        self.loss = 1e5
        self.flat_update_direction = None
    
    @torch.no_grad()
    def zero_grad(self):
        for p in self.params_list:
            if p.grad is not None:
                p.grad.zero_()

    def apply_preconditioner_to_grads(self):
        """
        Solve the least squares problem: x = argmin ||A*x - grads||^2
        and assign the solution back into each parameter's .grad.
        """
        # 1. Flatten all gradients into a single vector.
        grads = [p.grad for p in self.params_list if p.grad is not None]
        flat_grads = parameters_to_vector(grads)
        N = flat_grads.numel()
        eps = self.regularization
        A = compute_JTJ(self.params_dict, self.res_terms) + eps*torch.eye(N)

        # 2. Solve least squares: x = argmin_x ||A*x - grads||^2
        x, _, _, _ = torch.linalg.lstsq(A.double(), flat_grads.double(), driver="gels")
        x = x.to(flat_grads.dtype)
        self.flat_update_direction = x

        # 3. Unflatten x back into each parameter’s .grad
        vector_to_parameters(x, grads)
    
    @torch.no_grad()
    def step(self):
        """
        Perform a single Gauss-Newton update step on all parameters:
        """
        self.t += 1

        self.apply_preconditioner_to_grads()

        # if not self.do_line_search:
        #     for param in self.params_list:
        #         if param.grad is None:
        #             continue
        #         param.data -= self.lr * param.grad
        # If line search is enabled
        # TODO: do line search like in GaussNewtonNew, seems to work better
        if self.do_line_search:
            # current_loss = compute_loss(self.params_dict, self.config) # old
            current_loss = compute_loss(self.params_dict, self.res_terms)
            best_loss = current_loss
            best_lr = self.lr
            original_params = [param.clone() for param in self.params_list]
            best_params = original_params
            lrs = torch.logspace(0, -3, steps=self.line_search_steps)
        
            for lr in lrs:
                for param in self.params_list:
                    if param.grad is not None:
                        # param.data -= lr * param.grad
                        param.add_(param.grad, alpha=-lr)
                        
        
                params_dict = dict(zip(self.params_dict.keys(), self.params_list))
                # new_loss = compute_loss(params_dict, self.config) # old
                new_loss = compute_loss(params_dict, self.res_terms)
        
                if new_loss < best_loss:
                    best_loss = new_loss
                    best_lr = lr
                    best_params = [param.clone() for param in self.params_list]
                else:
                    for i, param in enumerate(original_params):
                        self.params_list[i].data.copy_(param.data)
        
            self.lr = best_lr
            for i, param in enumerate(best_params):
                self.params_list[i].data.copy_(param.data)
        else:
            for param in self.params_list:
                if param.grad is None:
                    continue
                # param.data -= self.lr * param.grad
                param.add_(param.grad, alpha=-self.lr)

In [ ]:
model = model.double()
params = model.params
loss_over_time = {}
distance_over_time = {}
chamfer_over_time = {}
best_loss = float('inf')

optim = GaussNewton(model, res_terms, lr=1e-1, regularization=1e-6, do_line_search=False)
start_time = time.time()
current_time = 0

for i in (pbar:=trange(100000)):
    if current_time > 1200:
        break
    optim.zero_grad()


    ## Update weights
    if i == 500:
        loss_weights = {"data": 1.0, "eikonal": 0.0, "mean_curvature": 1.0}
        for key in res_terms:
            res_terms[key].weight = loss_weights[key]

    ## Update surface points
    pts_surface = sample_model_surface_binsearch(model, pts_boundary, bound_limit=2)
    res_terms["mean_curvature"].points = pts_surface
    
    ## Evaluate the losses
    loss, unweighted_losses = compute_loss(params, res_terms, return_unweighted_losses=True)
        
    loss.backward()
    
    with torch.no_grad():
        loss_metric = unweighted_losses["data"] + unweighted_losses["mean_curvature"]

        current_time = time.time() - start_time
        loss_over_time[current_time] = loss_metric.item()
        distance_over_time[current_time] = compute_distance(model.double(), catenoid_level_set, pts_surface, pts_surface_true, 1.0)
        chamfer_over_time[current_time] = chamfer_div(model, pts_surface_true)

        if loss_metric.item() < best_loss:
            best_loss = loss_metric.item()
            best_model_state = copy.deepcopy(model.state_dict())

        unweighted_losses_str = " ".join(f"{key}: {l.item():.2e}" for key, l in unweighted_losses.items())
        pbar.set_description(unweighted_losses_str + " "
                            f"error: {chamfer_over_time[current_time]:.2e} "
                            f"nof_pts: {len(pts_surface)}"
                            )
    optim.step()

# Optional: Load the best model after training
# if best_model_state is not None:
#     model.load_state_dict(best_model_state)
#     print(f"Best model loaded with loss {best_loss}")

plt.plot(loss_over_time.keys(), loss_over_time.values())
plt.semilogy()
plt.show()

### Visualize the result

In [ ]:
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-2, -2, -1.25], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 1.25], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(pts_surface_true.cpu().detach(), point_size=0.05)

fig.display()

In [ ]:
verts, faces = get_mesh(
    model.float(), N=256, device=device,
    bbox_min=torch.tensor([-2, -2, -1.25], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 1.25], dtype=torch.float64),
    chunks=2
)

model.double()
mean_curvatures = r_mean_curvature(params, torch.tensor(verts, dtype=torch.float64)).squeeze(1).abs()

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)

color_map = k3d.basic_color_maps.Jet
color_range = [0, 0.005]

fig += k3d.mesh(
    verts, faces, 
    attribute=mean_curvatures.cpu().detach().numpy().astype(np.float32),
    color_range=color_range,
    color_map=color_map,
    side='double',
    flat_shading=False
)

fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

fig.display()